# 06 Segment Prediction


In [ ]:
from pathlib import Path
import sys

def find_project_root(start: Path) -> Path:
    markers = ("src", "data", "notebooks")

    for p in [start, *start.parents]:
        if p.name == "customer-segmentation-analytics" and all((p / m).is_dir() for m in markers):
            return p

    for p in [start, *start.parents]:
        candidate = p / "Improvements" / "Statistics" / "customer-segmentation-analytics"
        if all((candidate / m).is_dir() for m in markers):
            return candidate

    raise RuntimeError("Could not locate customer-segmentation-analytics project root.")

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"PROJECT_ROOT: {PROJECT_ROOT}")


## Objective


Train supervised models to predict `customer_segment` from customer behaviour features.


In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from src.modelling import build_segment_logistic_model, build_segment_random_forest

features = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "customer_features.parquet")
features.head()


## Build Training Matrix


In [ ]:
feature_cols = [
    "age",
    "annual_income",
    "months_active",
    "avg_monthly_spend",
    "purchase_frequency",
    "avg_order_value",
    "discount_usage_rate",
    "return_rate",
    "browsing_time_minutes",
    "support_interactions",
    "annualized_spend",
    "spend_to_income_ratio",
    "support_intensity",
    "discounted_order_value",
    "engagement_score",
    "payment_method",
    "region",
    "value_tier",
]

target_col = "customer_segment"

X = features[feature_cols].copy()
y = features[target_col].copy()

numeric_cols = X.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = [c for c in X.columns if c not in numeric_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


## Model 1: Multinomial Logistic Regression


In [ ]:
log_model = Pipeline(
    steps=[
        ("prep", preprocessor),
        ("model", build_segment_logistic_model(random_state=42)),
    ]
)

log_model.fit(X_train, y_train)
log_pred = log_model.predict(X_test)

log_results = {
    "accuracy": float(accuracy_score(y_test, log_pred)),
    "macro_f1": float(f1_score(y_test, log_pred, average="macro")),
}
log_results


## Model 2: Random Forest


In [ ]:
rf_model = Pipeline(
    steps=[
        ("prep", preprocessor),
        ("model", build_segment_random_forest(random_state=42)),
    ]
)

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

rf_results = {
    "accuracy": float(accuracy_score(y_test, rf_pred)),
    "macro_f1": float(f1_score(y_test, rf_pred, average="macro")),
}
rf_results


## Classification Report (Random Forest)


In [ ]:
print(classification_report(y_test, rf_pred))


## Export Prediction Scores


In [ ]:
proba = rf_model.predict_proba(X)
classes = rf_model.classes_

scores = pd.DataFrame(proba, columns=[f"prob_{c}" for c in classes])
scores.insert(0, "customer_id", features["customer_id"].values)
scores.insert(1, "predicted_segment", rf_model.predict(X))

out_scores = PROJECT_ROOT / "outputs" / "tables" / "segment_prediction_scores.csv"
out_scores.parent.mkdir(parents=True, exist_ok=True)
scores.to_csv(out_scores, index=False)
out_scores


## Note


This dataset does not include a true churn label. Any churn-risk extension should be explicitly defined as a proxy target.
